In [ ]:
from folktables import ACSDataSource, ACSIncome
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap as shap
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Get ACS data for Arizona, 2018
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person')
acs_data = data_source.get_data(states=["AZ"], download=True)

# Extract features and labels using folktables schema
features, labels, _ = ACSIncome.df_to_numpy(acs_data)

# Turn into DataFrame for readability
X = pd.DataFrame(features, columns=ACSIncome.features)
y = pd.Series(labels)

In [ ]:
# EDA for X
print(X.describe())


In [ ]:
# Feature names
feature_names = X.columns.tolist()
print("Feature names:", feature_names)

| Column Name | Description |
|-------------|-------------|
| `AGEP`      | Age of the person |
| `COW`       | Class of worker (e.g., private, government, self-employed) |
| `SCHL`      | Highest education level completed |
| `MAR`       | Marital status |
| `OCCP`      | Occupation code |
| `POBP`      | Place of birth (e.g., US state or foreign country) |
| `RELP`      | Relationship to the head of household |
| `WKHP`      | Usual hours worked per week |
| `SEX`       | Sex (0 = male, 1 = female in Folktables) |
| `RAC1P`     | Race (coded — requires mapping for labels) |

In [ ]:
X.columns = [
    "Age", "Class of Work", "Education Level", "Marital Status",
    "Occupation", "Birthplace", "Household Role", "Hours/Week",
    "Sex", "Race"
]

In [ ]:
print(y.value_counts())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=20, max_depth=8, random_state=0)
model.fit(X_train, y_train)

In [ ]:
# this taks a long time to run
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)


In [ ]:

# Summary plot
shap.summary_plot(shap_values, X_test)



In [ ]:
# what are other shap components?
shap.summary_plot(shap_values, X_test, plot_type="bar")


In [ ]:
shap.dependence_plot("SCHL", shap_values, X_test)

In [ ]:
shap.dependence_plot("AGEP", shap_values, X_test)


In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(n_estimators=50)
model.fit(X_train, y_train)

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)